# Лабораторная работа 2

### Задача 1

Исходные данные ([Kaggle](https://www.kaggle.com/datasets/mexwell/smart-home-energy-consumption/data)):

Основной файл smart_home_energy_consumption.csv содержит колонки:

- Home ID – идентификатор дома
- Appliance Type – тип прибора
- Energy Consumption (kWh) – потребление энергии (кВт·ч)
- Time – время (HH:MM)
- Date – дата (YYYY-MM-DD)
- Outdoor Temperature (°C) – температура наружного воздуха
- Season – сезон (Winter, Spring, Summer, Fall)
- Household Size – количество жильцов

Данные - синтетические

- Создайте датафрейм energy, прочитав файл "smart_home_energy_consumption.csv". Приведите столбец `Date` к типу datetime.

- Используя словарь `co2_emissions`, создайте новый столбец `CO2_per_kWh`, который для каждой строки содержит соответствующий коэффициент выбросов для типа прибора (`Appliance Type`):

    - Создайте пустой список или массив NumPy той же длины, что и DataFrame.
    - Пройдите по столбцу `Appliance Type` и для каждого значения найдите коэффициент в словаре.
    - Присвойте полученный список новому столбцу `energy['CO2_per_kWh']`.
    - Добавьте столбец `CO2_emissions` (`Energy Consumption (kWh)` * `CO2_per_kWh`).

Выведите первые 5 строк energy с помощью `head()`.

In [59]:
import pandas as pd
import numpy as np
energy = pd.read_csv(filepath_or_buffer= r"D:\Repos\Data_analysis_with_Python\smart_home_energy_consumption.csv", parse_dates=[4])
co2_per_kwh = np.ndarray(energy.shape[0])

In [60]:
co2_emissions = {
    'Air Conditioning': {'co2_per_kwh': 0.30, 'category': 'high', 'typical_power_w': 2500},
    'Computer': {'co2_per_kwh': 0.05, 'category': 'low', 'typical_power_w': 150},
    'Dishwasher': {'co2_per_kwh': 0.18, 'category': 'medium', 'typical_power_w': 1200},
    'Fridge': {'co2_per_kwh': 0.15, 'category': 'medium', 'typical_power_w': 150},
    'Heater': {'co2_per_kwh': 0.25, 'category': 'high', 'typical_power_w': 2000},
    'Lights': {'co2_per_kwh': 0.02, 'category': 'low', 'typical_power_w': 10},
    'Microwave': {'co2_per_kwh': 0.10, 'category': 'low', 'typical_power_w': 1000},
    'Oven': {'co2_per_kwh': 0.20, 'category': 'medium', 'typical_power_w': 2400},
    'TV': {'co2_per_kwh': 0.04, 'category': 'low', 'typical_power_w': 100},
    'Washing Machine': {'co2_per_kwh': 0.22, 'category': 'medium', 'typical_power_w': 500}
}

In [61]:
for ind, value in enumerate(energy["Appliance Type"]):
    co2_per_kwh[ind] = co2_emissions[value]["co2_per_kwh"]

energy["CO2_per_kWh"] = co2_per_kwh
energy["CO2_emissions"] = energy["CO2_per_kWh"] * energy["Energy Consumption (kWh)"]

energy.head(5)

,Home ID,Appliance Type,Energy Consumption (kWh),Time,Date,Outdoor Temperature (°C),Season,Household Size,CO2_per_kWh,CO2_emissions
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2,0.15,0.0300
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5,0.20,0.0460
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3,0.18,0.0576
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1,0.25,0.9800
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5,0.10,0.0440


### Задача 2. Энергоэффективность: потребление на человека (добавление столбца)

Добавьте столбец `consumption_per_person` = потребление (`Energy Consumption (kWh)`) / размер домохозяйства (`Household Size`).

Сравните среднее потребление на человека в небольших домохозяйствах (1-2 человека) и более крупных (3-5 человек).

Выясните, в каких домах выше потребление на человека.
Выведите соответствующие средние.



In [62]:
# YOUR CODE HERE
energy["consumption_per_person"] = energy["Energy Consumption (kWh)"]  / energy["Household Size"] 
energy.head(5)

small = np.mean(energy["consumption_per_person"][(energy["Household Size"] >= 1) & (energy["Household Size"] <= 2)])
big = np.mean(energy["consumption_per_person"][(energy["Household Size"] >= 3) & (energy["Household Size"] <= 5)])
print(f"Среднее потребление на человека в малых хозяйствах: {small}")
print(f"Среднее потребление на человека в больших хозяйствах: {big}")

Среднее потребление на человека в малых хозяйствах: 1.1366670429177015
Среднее потребление на человека в больших хозяйствах: 0.38977178088570336


### Задача 3. Фильтрация данных

Отфильтруйте energy (после добавления CO2) так, чтобы остались только записи, удовлетворяющие всем условиям:

- `Season` равен 'Summer' или 'Winter'
- `Energy Consumption (kWh)` > 2.0
- Если `Season` == 'Summer', то `Outdoor Temperature (°C)` > 30;
- если `Season` == 'Winter', то `Outdoor Temperature (°C)` < 0.

Результат сохраните в `filtered_energy`. Выведите размерность полученного после фильтрации датафрейма.

In [28]:
filtered_energy = energy[(energy["Energy Consumption (kWh)"] > 2.0) & (((energy["Season"] == "Summer") & (energy["Outdoor Temperature (°C)"]  > 30)) | ((energy["Season"] == "Winter") & (energy["Outdoor Temperature (°C)"]  < 0)))]
print(filtered_energy.shape)

(1992, 11)


### Задача 4. Группировка и агрегация

Сгруппируйте исходный датафрейм `energy` по `Appliance Type` и `Season`. Для каждой группы вычислите:

- суммарное потребление энергии,
- среднее и медианное потребление,
- количество записей
- процент от общего суммарного потребления

Результатом должен стать датафрейм с MultiIndex (`Appliance Type`, `Season`)


In [ ]:
# YOUR CODE HERE
all_consumption = energy["Energy Consumption (kWh)"].sum()

Appliance_Type_group = energy.groupby("Appliance Type")["Energy Consumption (kWh)"]
Appliance_Type_group_statistics = Appliance_Type_group.agg(['sum', 'mean', 'median', 'count'])
Appliance_Type_group_statistics["percent"] = Appliance_Type_group.sum() / all_consumption * 100
print(Appliance_Type_group_statistics)

Season_group = energy.groupby("Appliance Type")["Energy Consumption (kWh)"]
Season_group_statistics = Season_group.agg(['sum', 'mean', 'median', 'count'])
Season_group_statistics["percent"] = Season_group.sum() / all_consumption * 100
print(Season_group_statistics)


                       sum      mean  median  count    percent
Appliance Type                                                
Air Conditioning  35233.06  3.499857    3.49  10067  23.489465
Computer          10893.63  1.095498    1.09   9944   7.262654
Dishwasher        11138.51  1.103369    1.10  10095   7.425913
Fridge             2961.97  0.298255    0.30   9931   1.974710
Heater            34930.78  3.486802    3.48  10018  23.287938
Lights            11092.12  1.087356    1.08  10201   7.394985
Microwave         10951.26  1.100961    1.11   9947   7.301076
Oven              10963.51  1.103080    1.10   9939   7.309243
TV                10867.97  1.097221    1.10   9905   7.245547
Washing Machine   10962.35  1.101412    1.10   9953   7.308469
                       sum      mean  median  count    percent
Appliance Type                                                
Air Conditioning  35233.06  3.499857    3.49  10067  23.489465
Computer          10893.63  1.095498    1.09   9944   7

### Задача 5. Добавление столбца с помощью apply

Напишите функцию consumption_category(kwh), которая возвращает:

- 'low' при kwh < 1.0
- 'medium' при 1.0 ≤ kwh < 3.0
- 'high' при kwh ≥ 3.0

Примените её к столбцу `Energy Consumption (kWh)` исходного датафрейма `energy` и создайте новый столбец Category. Выведите процентное распределение категорий (долю от общего числа записей) с округлением до 2 знаков.

In [36]:
# YOUR CODE HERE
energy["Category"] = energy["Energy Consumption (kWh)"].apply(lambda x: "low" if x < 1 else "medium" if x < 3 else "high")
print(energy)
percetage = energy["Category"].value_counts() / energy["Category"].count()
print(percetage)

       Home ID    Appliance Type  Energy Consumption (kWh)   Time       Date  \
0           94            Fridge                      0.20  21:12 2023-12-02   
1          435              Oven                      0.23  20:11 2023-08-06   
2          466        Dishwasher                      0.32  06:39 2023-11-21   
3          496            Heater                      3.92  21:56 2023-01-21   
4          137         Microwave                      0.44  04:31 2023-08-26   
...        ...               ...                       ...    ...        ...   
99995      124         Microwave                      0.42  09:56 2023-09-28   
99996      184          Computer                      0.71  12:48 2023-05-27   
99997      101        Dishwasher                      0.25  05:45 2023-02-18   
99998      423  Air Conditioning                      2.69  12:39 2023-04-20   
99999      429            Fridge                      0.37  18:46 2023-02-27   

       Outdoor Temperature (°C)  Season

### Задача 6: Группировка домов по уровню энергопотребления

Сгруппируйте датафрейм `energy` по домам (`Home ID`), вычислите суммарное потребление каждого дома за все время (`Energy Consumption (kWh)`).

Разделите дома на 3 категории:

- 'low' – нижние 25%
- 'medium' – средние 50%
- 'high' – верхние 25%

Выведите количество домов в каждой категории и среднее потребление в категории.


In [48]:
home_groups = energy.groupby("Home ID")["Energy Consumption (kWh)"].sum()

categories = pd.qcut(home_groups, [0,0.25,0.75,1], ["low","medium","high"])
result = pd.DataFrame({"Consumption": home_groups, "Category" : categories })
print(f"low: {result["Consumption"][result["Category"] == "low"].mean()} " + 
      f"medium: {result["Consumption"][result["Category"] == "medium"].mean()} " +
      f"high: {result["Consumption"][result["Category"] == "high"].mean()}")


low: 266.52168 medium: 299.43352000000004 high: 334.57256000000007


### Задача 7. Структура энергопотребления в зависимости от времени суток

В датафрейме `energy` преобразуйте столбец `Time` в час (целое число от 0 до 23). Сгруппируйте по часу и найдите среднее потребление (`Energy Consumption (kWh)`).

- Выведите среднее потребление по часам (отсортированное по часу)
- Определите час с максимальным средним потреблением (пиковый) и час с минимальным.

In [58]:
# YOUR CODE HERE
#energy["Time"] = energy["Time"].apply(lambda x: int(x[0:2]))
groups_by_time = energy.groupby("Time")["Energy Consumption (kWh)"].mean()
print(f"максималное потребление в {groups_by_time.idxmax()} час : {groups_by_time.max()}")
print(f"минимальное потребление в {groups_by_time.idxmin()} час : {groups_by_time.min()}")

максималное потребление в 14 час : 1.5388007644529385
минимальное потребление в 23 час : 1.4697783541617544


### Задача 8. Resample

Используя датафрейм `energy` (из задачи 2), создайте сводную таблицу с суммарным потреблением энергии по `Date` и `Appliance Type`.

Используйте resample по неделям ('W') и вычислите среднее недельное потребление для каждого прибора.

На выходе должен получиться датафрейм с неделями в качестве индексе и приборами в колонках. Выведите для него описательные статистики через `describe()`.

In [70]:
pivot = energy.pivot_table(
    values="Energy Consumption (kWh)",
    index='Date',
    columns='Appliance Type',
    aggfunc='sum',
    fill_value=0
)
weekly_consumption = pivot.resample('W').mean()
print(weekly_consumption.describe())

Appliance Type  Air Conditioning   Computer  Dishwasher     Fridge  \
count                  54.000000  54.000000   54.000000  54.000000   
mean                   96.528836  29.936111   30.257116   8.149392   
std                     7.278616   2.881532    2.553168   0.719304   
min                    81.135714  24.405714   22.400000   6.620000   
25%                    91.723571  28.462143   28.454643   7.702857   
50%                    96.539286  29.745714   30.162143   8.080000   
75%                   101.989643  31.341071   32.169286   8.406071   
max                   108.270000  43.790000   37.284286  10.220000   

Appliance Type      Heater     Lights  Microwave       Oven         TV  \
count            54.000000  54.000000  54.000000  54.000000  54.000000   
mean             95.887090  30.351058  29.904127  29.891455  29.712354   
std               8.793421   2.513196   2.598182   2.480255   2.754127   
min              80.437143  24.385714  24.644286  24.622857  23.240000   

### Задача 9. Объединение с дополнительным датафреймом (2 балла)

Загрузите `United_States_weather_data.csv` (Источник: [Kaggle](https://www.kaggle.com/datasets/shivamshinde1904/weather-data2000-2023?select=United_States_weather_data.csv)), приведите `Date` к типу datetime. Объедините `energy` (из задачи 2) с `weather` по полю `Date` (left join).

1. Сравнение температур
В объединённом датафрейме сравните показания наружного датчика (`Outdoor Temperature (°C)`) с метеоданными (`Temp_Mean`, `Temp_Min`, `Temp_Max`).

Для каждой строки вычислите отклонения:

- модуль разницы между `Outdoor Temperature (°C)` и метеоданными (`Temp_Mean`, `Temp_Min`, `Temp_Max`)
- найдите среднее отклонение для каждого типа (среднее, минимум, максимум) по всем записям

И / ИЛИ:

2. Влияние солнечной активности на энергопотребление

Разделите все записи на 3 группы по продолжительности солнечного сияния (`Sunshine_Duration`) как в задаче 7:

- 'low' – нижние 33% значений
- 'medium' – средние 34%
- 'high' – верхние 33%

Для каждой группы вычислите средние `Energy Consumption (kWh)` и `CO2_emissions`.



In [89]:
weather = pd.read_csv(r'D:\Repos\Data_analysis_with_Python\United_States_weather_data.csv', parse_dates =[1])
weather["Date"] = pd.to_datetime(weather["Date"].apply(lambda x: x.split("-")[1] + "-" + x.split("-")[0] + "-" + x.split("-")[2] ))

merged = pd.merge(energy,weather,how="left", on= "Date")
merged["Temp_mean_difference"] =  (merged["Outdoor Temperature (°C)"] - merged["Temp_Mean"]).abs() 
merged["Temp_min_difference"] =  (merged["Outdoor Temperature (°C)"] - merged["Temp_Min"]).abs() 
merged["Temp_max_difference"] =  (merged["Outdoor Temperature (°C)"] - merged["Temp_Max"]).abs() 

print(f"Среднее отклонение среднего: {merged["Temp_mean_difference"].mean()}\n" +
      f"Среднее отклонение минимального: {merged["Temp_min_difference"].mean()}\n" +
      f"Среднее отклонение максимального: {merged["Temp_max_difference"].mean()}\n")


categories = pd.qcut(merged["Sunshine_Duration"], [0,0.33,0.67,1], ["low","medium","high"])
result = pd.DataFrame({"Energy Consumption (kWh)": merged["Energy Consumption (kWh)"], "CO2_emissions" : merged["CO2_emissions"],  "Category" : categories })

print(f"низкая солнечность среднее потребление {result[result["Category"] == "low"]["Energy Consumption (kWh)"].mean()} средние выбросы {result[result["Category"] == "low"]["CO2_emissions"].mean()}")
print(f"средняя солнечность среднее потребление {result[result["Category"] == "medium"]["Energy Consumption (kWh)"].mean()} средние выбросы {result[result["Category"] == "medium"]["CO2_emissions"].mean()}")
print(f"высокая солнечность среднее потребление {result[result["Category"] == "high"]["Energy Consumption (kWh)"].mean()} средние выбросы {result[result["Category"] == "high"]["CO2_emissions"].mean()}")

Среднее отклонение среднего: 14.286118855171585
Среднее отклонение минимального: 14.470155839467298
Среднее отклонение максимального: 15.371821536733586

низкая солнечность среднее потребление 1.4966970276588638 средние выбросы 0.2861602513890154
средняя солнечность среднее потребление 1.4954138623214075 средние выбросы 0.2848704835281615
высокая солнечность среднее потребление 1.508603251785443 средние выбросы 0.28877011396444313
